In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Tb₂Ti₂O₇ — neutron single crystal, constant wavelength, isotropic extinction

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='tbti')

structure.space_group.name_h_m = 'F d -3 m'  # FullProf Space group symbol

structure.cell.length_a = 10.130  # FullProf a

# Anisotropic sites carry the FullProf β tensor directly: ``adp_type`` is
# set to ``'beta'`` and the dimensionless β components are assigned
# verbatim. F d -3 m is cubic, so symmetry links the remaining β
# components; only the independent ones are set. FullProf occupancy folds
# in the site multiplicity; CIF/EasyDiffraction use 1.0 for a fully
# occupied site.
structure.atom_sites.create(
    id='Tb',  # FullProf Atom
    type_symbol='Tb',  # FullProf Typ
    fract_x=0.5,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['Tb']
aniso.adp_11 = 0.00098991673  # FullProf beta11
aniso.adp_12 = -0.00047650724  # FullProf beta12

structure.atom_sites.create(
    id='Ti',  # FullProf Atom
    type_symbol='Ti',  # FullProf Typ
    fract_x=0,  # FullProf X
    fract_y=0,  # FullProf Y
    fract_z=0,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['Ti']
aniso.adp_11 = 0.00090989727  # FullProf beta11
aniso.adp_12 = -0.00016990340  # FullProf beta12

structure.atom_sites.create(
    id='O1',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.32804,  # FullProf X
    fract_y=0.125,  # FullProf Y
    fract_z=0.125,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['O1']
aniso.adp_11 = 0.0012294180  # FullProf beta11
aniso.adp_22 = 0.00078215479  # FullProf beta22
aniso.adp_23 = 0.00041246481  # FullProf beta23

structure.atom_sites.create(
    id='O2',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.375,  # FullProf X
    fract_y=0.375,  # FullProf Y
    fract_z=0.375,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['O2']
aniso.adp_11 = 0.00060762477  # FullProf beta11

project.structures.add(structure)

In [5]:
structure.show_as_text()

Structure 🧩 'tbti' as text


,CIF
1,data_tbti
2,
3,_cell.length_a 10.13
4,_cell.length_b 10.13
5,_cell.length_c 10.13
6,_cell.angle_alpha 90.
7,_cell.angle_beta 90.
8,_cell.angle_gamma 90.
9,
10,"_space_group.name_h_m ""F d -3 m"""


## Load the FullProf reference

In [6]:
FULLPROF_PROJECT_DIR = 'sc-neut-cwl_ext-iso_tbti'
FULLPROF_OUT_FILE = 'tbti.out'
FULLPROF_SCALE = 0.37517014  # FullProf Scale
FULLPROF_WAVELENGTH = 0.7930  # FullProf Lambda
# cryspy uses Becker-Coppens isotropic extinction, not the one from
# FullProf
EXTINCTION_RADIUS = 10.0
EXTINCTION_MOSAICITY = 35000.0

f2calc = verify.load_fullprof_sc_f2calc(FULLPROF_PROJECT_DIR, FULLPROF_OUT_FILE)

## Create the experiment

In [7]:
experiment = ExperimentFactory.from_scratch(
    name='tbti',
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment.linked_structure.structure_id = 'tbti'
experiment.linked_structure.scale = FULLPROF_SCALE
experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.extinction.type = 'becker-coppens'
experiment.extinction.model = 'gauss'
experiment.extinction.radius.value = EXTINCTION_RADIUS
experiment.extinction.mosaicity.value = EXTINCTION_MOSAICITY

verify.set_reference_reflections(experiment, f2calc)

project.experiments.add(experiment)

Extinction type changed to


becker-coppens


## edi-cryspy VS FullProf

In [8]:
calc_ed_cryspy = verify.calculate_reflections(project, experiment, 'cryspy')
reference, candidate = verify.align_reflections(f2calc, calc_ed_cryspy)

project.display.reflection_comparison(
    'tbti',
    reference=reference,
    candidate=candidate,
    reference_label='FullProf',
    candidate_label='edi-cryspy',
)

Calculator for experiment 'tbti' already set to


cryspy


## Fit edi-cryspy to FullProf

In [9]:
experiment.calculator.type = 'cryspy'

experiment.linked_structure.scale.free = True
experiment.extinction.radius.free = True

project.analysis.fit()
project.display.fit.results()

calc_ed_cryspy_refined = verify.calculate_reflections(project, experiment, 'cryspy')
reference_refined, candidate_refined = verify.align_reflections(f2calc, calc_ed_cryspy_refined)

project.display.reflection_comparison(
    'tbti',
    reference=reference_refined,
    candidate=candidate_refined,
    reference_label='FullProf',
    candidate_label='edi-cryspy (scale + ext radius)',
)

verify.report_refinement_closeness(
    reference,
    candidate,
    candidate_refined,
)

Calculator for experiment 'tbti' already set to


cryspy


<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'tbti' for 'single' fitting


...

📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,tbti,extinction,,radius,μm,10.0000,22.3613,0.4183,123.61 % ↑
2,tbti,linked_structure,,scale,,0.3752,2.6177,0.0189,597.73 % ↑


Calculator for experiment 'tbti' already set to


cryspy


,Metric,Before,After
1,Profile diff (%),83.03,4.19
2,Max deviation (%),82.38,6.48
3,Area ratio,0.1625,1.0014
4,Shape correlation,0.9952,0.9986


## Agreement check

In [10]:
verify.assert_patterns_agree(
    [
        ('cryspy vs FullProf', reference_refined, candidate_refined),
    ],
    raise_on_failure=False,
)

,Comparison,Metric,Expected,Actual,OK
1,cryspy vs FullProf,Profile diff (%),< 2.5,4.19,❌
2,,Max deviation (%),< 6,6.48,❌
3,,Area ratio,0.99 to 1.01,1.0014,✅
4,,Shape correlation,> 0.999,0.9986,❌


False